# 할 일 목록을 관리하는 챗봇 (State 활용)

이번엔 LLM 없이 **State 설계 자체**에 집중한다. 사용자 입력과 플래그(추가/완료)에 따라 할 일 목록을 갱신하는 그래프를 만든다.

포인트: `todo_list` 는 매번 통째로 갱신(덮어쓰기)하고, `completed_list` 는 `operator.add` 리듀서로 **누적**한다 — 같은 State 안에서 필드마다 갱신 방식을 다르게 가져갈 수 있다.

## Step 1. State 정의하기

In [ ]:
from typing import Annotated
from operator import add
from typing_extensions import TypedDict
from langgraph.graph import StateGraph

class TodoState(TypedDict):
    user_input: str
    flag: str                                  # 'add'(추가) 또는 'complete'(완료)
    todo_list: list[str]                       # 덮어쓰기로 관리
    completed_list: Annotated[list[str], add]  # 리듀서로 누적

## Step 2. 할 일 추가/완료 노드

- `flag == 'add'` : 입력을 `/` 로 나눠 여러 할 일을 한 번에 추가
- `flag == 'complete'` : 해당 할 일을 목록에서 제거하고 완료 목록에 누적

In [ ]:
def update_todo_list(state: TodoState) -> TodoState:
    user_input, flag = state["user_input"], state["flag"]
    todo_list = state["todo_list"]

    if flag == "add":
        todo_list.extend(user_input.split("/"))
    elif flag == "complete":
        if user_input in todo_list:
            todo_list.remove(user_input)

    return {
        "user_input": user_input,
        "flag": flag,
        "todo_list": todo_list,
        # 완료일 때만 누적 리스트에 추가됨 (리듀서 add)
        "completed_list": [user_input] if flag == "complete" else [],
    }

## Step 3. 그래프 생성

In [ ]:
graph_builder = StateGraph(TodoState)
graph_builder.add_node("update_todo", update_todo_list)
graph_builder.set_entry_point("update_todo")

graph = graph_builder.compile()

In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    print(graph.get_graph().draw_mermaid())

## Step 4. 실행해보기

`invoke` 를 여러 번 호출하며 직전 결과를 다음 입력의 초기 상태로 넘긴다.
여기서는 입력 시나리오를 미리 정해 흐름을 보여준다 (실제로는 `input()` 으로 받아도 된다).

In [ ]:
# (user_input, flag) 시나리오
scenario = [
    ("우유 사기/운동하기/책 읽기", "add"),
    ("운동하기", "complete"),
    ("청소하기", "add"),
    ("우유 사기", "complete"),
]

state = TodoState(user_input="", flag="", todo_list=[], completed_list=[])
for user_input, flag in scenario:
    state = graph.invoke({
        "user_input": user_input,
        "flag": flag,
        "todo_list": state["todo_list"],
        "completed_list": state["completed_list"],
    })
    print(f"[{flag}] {user_input!r}")
    print("  할 일:", state["todo_list"])
    print("  완료:", state["completed_list"])
    print()

직접 입력하며 돌리고 싶다면 아래처럼 `input()` 루프를 쓴다.

```python
state = TodoState(user_input="", flag="", todo_list=[], completed_list=[])
while True:
    user_input = input("User: ")
    if user_input.lower() in ["quit", "exit", "q"]:
        break
    flag = input("Flag(add/complete): ")
    state = graph.invoke({
        "user_input": user_input, "flag": flag,
        "todo_list": state["todo_list"],
        "completed_list": state["completed_list"],
    })
    print(state)
```

## 정리

- 한 State 안에서 필드마다 갱신 방식을 다르게: `todo_list`(덮어쓰기) vs `completed_list`(리듀서 누적)
- 그래프를 반복 `invoke` 하며 직전 상태를 넘겨 상태를 이어간다

다음: LLM 응답을 정해진 데이터 구조로 받는 **Structured Output**.